# 🚀 fixed SadTalker Video Generator (TalentForge Custom Edition)
This notebook contains the complete fixes for the **numpy attribute errors** and **HuggingFace 403 Forbidden** download errors. 

In [ ]:
# 1. Downgrade numpy to a version compatible with SadTalker
!pip install "numpy<2.0"
print("\n✓ Numpy downgraded successfully!")
print("⚠️ IMPORTANT: If Colab shows a 'Restart Session' popup, click it! Then run the next cell below.")

In [ ]:
# 2. Clone SadTalker repository and install UI dependencies
%cd /content
!git clone -b v1.1 https://github.com/camenduru/SadTalker
!pip install -q gradio safetensors kornia facexlib yacs gfpgan
%cd /content/SadTalker

# 3. Download weights using requests with a browser User-Agent (bypasses 403 Forbidden blocks)
import os
import requests
from tqdm import tqdm

def download_file(url, dest_folder):
    os.makedirs(dest_folder, exist_ok=True)
    filename = url.split('/')[-1]
    dest_path = os.path.join(dest_folder, filename)
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    print(f"Downloading {filename}...")
    response = requests.get(url, stream=True, headers=headers)
    if response.status_code != 200:
        print(f"ERROR: Failed to download {filename} (HTTP Status {response.status_code})")
        return False
        
    total_size = int(response.headers.get('content-length', 0))
    with open(dest_path, 'wb') as file, tqdm(
        desc=filename,
        total=total_size,
        unit='B',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(chunk_size=1024*1024):
            size = file.write(data)
            bar.update(size)
    print(f"✓ Saved to {dest_path}\n")
    return True

print("\n--- Downloading Checkpoints ---")
checkpoint_urls = [
    "https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/SadTalker_v0.0.2_256.safetensors",
    "https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/SadTalker_v0.0.2_512.safetensors",
    "https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/mapping_00109-model.pth.tar",
    "https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/mapping_00229-model.pth.tar"
]
for url in checkpoint_urls:
    download_file(url, "/content/SadTalker/checkpoints")

print("--- Downloading GFPGAN Face Restoration Model ---")
gfpgan_urls = [
    "https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/GFPGANv1.4.pth",
    "https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/alignment_WFLW_4HG.pth",
    "https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/detection_Resnet50_Final.pth",
    "https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/parsing_parsenet.pth"
]
for url in gfpgan_urls:
    download_file(url, "/content/SadTalker/gfpgan/weights")

print("\n🚀 Starting SadTalker WebUI...")
!python app_sadtalker.py